In [0]:
%pip install -q catboost

In [0]:
%restart_python

In [0]:
import catboost

print(catboost.__version__)

In [0]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.catboost

from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from mlflow.models import infer_signature

from pyspark.sql import functions as F

In [0]:
SOURCE_TABLE = "high_garden.gold.forecasting_features"

PREDICTIONS_TABLE = (
    "high_garden.gold.backtest_predictions"
)

METRICS_TABLE = (
    "high_garden.gold.model_metrics"
)

YEARLY_TABLE = (
    "high_garden.gold.yearly_model_performance"
)

EXPERIMENT_NAME = (
    "/Shared/high-garden-coffee"
)

In [0]:
sdf = spark.table(
    SOURCE_TABLE
)

print(
    "Rows:",
    sdf.count()
)

display(
    sdf.limit(10)
)

In [0]:
sdf = spark.table(
    SOURCE_TABLE
)

print(
    "Rows:",
    sdf.count()
)

display(
    sdf.limit(10)
)

In [0]:
pdf = sdf.toPandas()

print(
    "Shape:",
    pdf.shape
)

print(
    pdf.columns.tolist()
)

In [0]:
categorical_features = [
    "country",
    "coffee_type",
]

numeric_features = [
    "start_year",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_5",
    "rolling_mean_3",
    "rolling_mean_5",
    "rolling_std_3",
    "historical_growth_1y",
    "lag_1_zero",
]

feature_columns = (
    categorical_features
    +
    numeric_features
)

target_column = "target"

In [0]:
for column in categorical_features:
    pdf[column] = (
        pdf[column]
        .astype(str)
    )

In [0]:
required_history = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_5",
    "rolling_mean_3",
    "rolling_mean_5",
]

model_df = (
    pdf
    .dropna(
        subset=
        required_history
        +
        [target_column]
    )
    .copy()
)

In [0]:
print(
    "Original rows:",
    len(pdf)
)

print(
    "Model-ready rows:",
    len(model_df)
)

print(
    "Available years:",
    sorted(
        model_df[
            "start_year"
        ].unique()
    )
)

In [0]:
assert (
    model_df[target_column]
    .isna()
    .sum()
    == 0
), "Null targets detected"

assert (
    model_df[
        target_column
    ].lt(0).sum()
    == 0
), "Negative targets detected"

assert (
    len(model_df) > 0
), "No training rows available"

print(
    "Model dataset validation passed."
)

In [0]:
BACKTEST_YEARS = [
    2015,
    2016,
    2017,
    2018,
    2019,
]

In [0]:
def calculate_metrics(
    y_true,
    y_pred,
    mase_scale=None,
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    denominator = (
        np.abs(y_true)
        .sum()
    )

    if denominator > 0:

        wape = (
            np.abs(
                y_true
                -
                y_pred
            ).sum()
            /
            denominator
        )

    else:

        wape = np.nan

    if (
        mase_scale is not None
        and
        mase_scale > 0
    ):

        mase = (
            mae
            /
            mase_scale
        )

    else:

        mase = np.nan

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "wape": float(wape),
        "mase": float(mase),
    }

In [0]:
mlflow.set_experiment(
    EXPERIMENT_NAME
)

In [0]:
naive_fold_metrics = []
naive_predictions = []

for test_year in BACKTEST_YEARS:

    train_df = model_df[
        model_df["start_year"]
        < test_year
    ].copy()

    test_df = model_df[
        model_df["start_year"]
        == test_year
    ].copy()

    assert (
        len(train_df) > 0
    )

    assert (
        len(test_df) > 0
    )

    y_true = (
        test_df[
            target_column
        ].values
    )

    y_pred = (
        test_df[
            "lag_1"
        ].values
    )

    mase_scale = np.mean(
        np.abs(
            train_df[
                target_column
            ]
            -
            train_df[
                "lag_1"
            ]
        )
    )

    metrics = calculate_metrics(
        y_true,
        y_pred,
        mase_scale,
    )

    metrics[
        "year"
    ] = test_year

    naive_fold_metrics.append(
        metrics
    )

    fold_predictions = (
        test_df[
            [
                "country",
                "coffee_type",
                "start_year",
                target_column,
            ]
        ]
        .copy()
    )

    fold_predictions[
        "raw_prediction"
    ] = y_pred

    fold_predictions[
        "prediction"
    ] = y_pred

    fold_predictions[
        "model"
    ] = "naive"

    naive_predictions.append(
        fold_predictions
    )

In [0]:
naive_metrics_df = (
    pd.DataFrame(
        naive_fold_metrics
    )
)

display(
    naive_metrics_df
)

In [0]:
naive_summary = {
    "mae":
        naive_metrics_df[
            "mae"
        ].mean(),

    "rmse":
        naive_metrics_df[
            "rmse"
        ].mean(),

    "wape":
        naive_metrics_df[
            "wape"
        ].mean(),

    "mase":
        naive_metrics_df[
            "mase"
        ].mean(),

    "negative_raw_predictions":
        0,
}

print(
    naive_summary
)

In [0]:
with mlflow.start_run(
    run_name="naive_baseline"
):

    mlflow.log_param(
        "model_type",
        "naive_persistence"
    )

    mlflow.log_param(
        "validation",
        "rolling_origin"
    )

    mlflow.log_param(
        "backtest_start",
        min(BACKTEST_YEARS)
    )

    mlflow.log_param(
        "backtest_end",
        max(BACKTEST_YEARS)
    )

    for metric in [
        "mae",
        "rmse",
        "wape",
        "mase",
    ]:

        mlflow.log_metric(
            metric,
            float(
                naive_summary[
                    metric
                ]
            )
        )

    mlflow.set_tag(
        "stage",
        "baseline"
    )

In [0]:
CATBOOST_PARAMS = {
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "MAE",
    "random_seed": 42,
    "verbose": False,
}

In [0]:
catboost_fold_metrics = []
catboost_predictions = []

total_negative_raw = 0

for test_year in BACKTEST_YEARS:

    train_df = model_df[
        model_df["start_year"]
        < test_year
    ].copy()

    test_df = model_df[
        model_df["start_year"]
        == test_year
    ].copy()

    assert (
        len(train_df) > 0
    )

    assert (
        len(test_df) > 0
    )

    X_train = train_df[
        feature_columns
    ]

    y_train = train_df[
        target_column
    ]

    X_test = test_df[
        feature_columns
    ]

    y_test = test_df[
        target_column
    ]

    model = CatBoostRegressor(
        **CATBOOST_PARAMS
    )

    model.fit(
        X_train,
        y_train,
        cat_features=
        categorical_features,
    )

    # ---------------------
    # RAW MODEL PREDICTION
    # ---------------------

    y_pred_raw = model.predict(
        X_test
    )

    negative_raw_count = int(
        np.sum(
            y_pred_raw < 0
        )
    )

    total_negative_raw += (
        negative_raw_count
    )

    # ---------------------
    # BUSINESS CONSTRAINT
    # ---------------------

    y_pred = np.clip(
        y_pred_raw,
        a_min=0,
        a_max=None,
    )

    # ---------------------
    # MASE SCALE
    # ---------------------

    mase_scale = np.mean(
        np.abs(
            train_df[
                target_column
            ]
            -
            train_df[
                "lag_1"
            ]
        )
    )

    # ---------------------
    # METRICS
    # ---------------------

    metrics = calculate_metrics(
        y_test.values,
        y_pred,
        mase_scale,
    )

    metrics[
        "year"
    ] = test_year

    metrics[
        "negative_raw_predictions"
    ] = negative_raw_count

    catboost_fold_metrics.append(
        metrics
    )

    # ---------------------
    # SAVE PREDICTIONS
    # ---------------------

    fold_predictions = (
        test_df[
            [
                "country",
                "coffee_type",
                "start_year",
                target_column,
            ]
        ]
        .copy()
    )

    fold_predictions[
        "raw_prediction"
    ] = y_pred_raw

    fold_predictions[
        "prediction"
    ] = y_pred

    fold_predictions[
        "model"
    ] = "catboost"

    catboost_predictions.append(
        fold_predictions
    )

In [0]:
print(
    "Raw negative predictions:",
    total_negative_raw
)

In [0]:
catboost_metrics_df = (
    pd.DataFrame(
        catboost_fold_metrics
    )
)

display(
    catboost_metrics_df
)

In [0]:
catboost_summary = {
    "mae":
        catboost_metrics_df[
            "mae"
        ].mean(),

    "rmse":
        catboost_metrics_df[
            "rmse"
        ].mean(),

    "wape":
        catboost_metrics_df[
            "wape"
        ].mean(),

    "mase":
        catboost_metrics_df[
            "mase"
        ].mean(),

    "negative_raw_predictions":
        int(
            catboost_metrics_df[
                "negative_raw_predictions"
            ].sum()
        ),
}

print(
    catboost_summary
)

In [0]:
comparison = pd.DataFrame(
    [
        {
            "model":
                "naive",

            **naive_summary,
        },

        {
            "model":
                "catboost",

            **catboost_summary,
        },
    ]
)

display(
    comparison
    .sort_values(
        "wape"
    )
)

In [0]:
X_full = model_df[
    feature_columns
]

y_full = model_df[
    target_column
]

final_model = CatBoostRegressor(
    **CATBOOST_PARAMS
)

final_model.fit(
    X_full,
    y_full,
    cat_features=
        categorical_features,
)

In [0]:
with mlflow.start_run(
    run_name="catboost_global"
) as run:

    for key, value in (
        CATBOOST_PARAMS.items()
    ):

        mlflow.log_param(
            key,
            value
        )

    mlflow.log_param(
        "model_type",
        "CatBoostRegressor"
    )

    mlflow.log_param(
        "validation",
        "rolling_origin"
    )

    mlflow.log_param(
        "folds",
        len(BACKTEST_YEARS)
    )

    mlflow.log_param(
        "non_negative_constraint",
        "clip_at_zero"
    )

    for metric in [
        "mae",
        "rmse",
        "wape",
        "mase",
    ]:

        mlflow.log_metric(
            metric,
            float(
                catboost_summary[
                    metric
                ]
            )
        )

    mlflow.log_metric(
        "negative_raw_predictions",
        float(
            catboost_summary[
                "negative_raw_predictions"
            ]
        )
    )

    input_example = (
        X_full
        .head(5)
    )

    output_example = (
        np.clip(
            final_model.predict(
                input_example
            ),
            a_min=0,
            a_max=None,
        )
    )

    signature = infer_signature(
        input_example,
        output_example,
    )

    mlflow.catboost.log_model(
        final_model,
        name="model",
        input_example=
            input_example,
        signature=
            signature,
    )

    mlflow.set_tag(
        "stage",
        "candidate"
    )

    mlflow.set_tag(
        "data_source",
        SOURCE_TABLE
    )

    catboost_run_id = (
        run.info.run_id
    )

In [0]:
print(
    "CatBoost MLflow run:",
    catboost_run_id
)

In [0]:
all_predictions = pd.concat(
    naive_predictions
    +
    catboost_predictions,
    ignore_index=True,
)

In [0]:
display(
    all_predictions.head(20)
)

In [0]:
assert (
    all_predictions[
        "prediction"
    ].lt(0).sum()
    == 0
), "Negative final predictions detected"

print(
    "Final prediction constraint passed."
)

In [0]:
raw_negative_summary = (
    all_predictions
    .groupby("model")
    ["raw_prediction"]
    .apply(
        lambda values:
        int(
            (values < 0).sum()
        )
    )
)

print(
    raw_negative_summary
)

In [0]:
predictions_sdf = (
    spark.createDataFrame(
        all_predictions
    )
)

(
    predictions_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        PREDICTIONS_TABLE
    )
)

In [0]:
comparison_sdf = (
    spark.createDataFrame(
        comparison
    )
)

(
    comparison_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        METRICS_TABLE
    )
)

In [0]:
negative_final_predictions = (
    spark.table(
        PREDICTIONS_TABLE
    )
    .filter(
        F.col("prediction") < 0
    )
    .count()
)

print(
    "Negative FINAL predictions:",
    negative_final_predictions
)

In [0]:
display(
    spark.table(
        PREDICTIONS_TABLE
    )
    .filter(
        (F.col("model") == "catboost")
        &
        (
            F.col(
                "raw_prediction"
            ) < 0
        )
    )
    .orderBy(
        "raw_prediction"
    )
)

In [0]:
yearly_model_performance = (
    spark.table(
        PREDICTIONS_TABLE
    )
    .groupBy(
        "model",
        "start_year"
    )
    .agg(
        F.sum(
            "target"
        ).alias(
            "actual"
        ),

        F.sum(
            "prediction"
        ).alias(
            "predicted"
        )
    )
    .withColumn(
        "absolute_error",
        F.abs(
            F.col("actual")
            -
            F.col("predicted")
        )
    )
    .withColumn(
        "percentage_error",
        F.when(
            F.col("actual") > 0,
            F.col(
                "absolute_error"
            )
            /
            F.col(
                "actual"
            )
        )
    )
    .orderBy(
        "start_year",
        "model"
    )
)

display(
    yearly_model_performance
)

In [0]:
(
    yearly_model_performance.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        YEARLY_TABLE
    )
)

In [0]:
catboost_plot = (
    spark.table(
        PREDICTIONS_TABLE
    )
    .filter(
        F.col("model")
        == "catboost"
    )
    .groupBy(
        "start_year"
    )
    .agg(
        F.sum(
            "target"
        ).alias(
            "actual"
        ),

        F.sum(
            "prediction"
        ).alias(
            "predicted"
        )
    )
    .orderBy(
        "start_year"
    )
)

display(
    catboost_plot
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table(
        METRICS_TABLE
    )
    .orderBy(
        "wape"
    )
)